In [1]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
import pickle
import matplotlib.pyplot as plt

## Data & preprocessing

In [2]:
# Load data
freq      = "seasonal"
df        = pd.read_csv(f'../data/covid_{freq}.csv', parse_dates=['date'])
block_cov = pd.read_csv(f'../data/block_cov_{freq}.csv')
block_cov = block_cov.set_index(['country', 'time_block'])

countries = df['country'].unique()
cont_map  = df.drop_duplicates('country').set_index('country')['continent'].to_dict()
cont_list = sorted(set(cont_map.values()))
cont_idx  = {c: i for i, c in enumerate(cont_list)}
n_cont    = len(cont_list)

all_blocks = sorted(df['time_block'].unique())
n_blocks   = len(all_blocks)
block_idx  = {b: i for i, b in enumerate(all_blocks)}

sigma2 = 0.03


## Model formulation

**[A] Regularized seasonal drift:**

$$
\mu_{t,c}
=
X_{t,c}^\top(\beta + \delta_{s(t)})
+
\theta_{\mathrm{cont}(c)},
\qquad
X_{t,c}
=
(1,U_{c1},U_{c2},V_{\mathrm{str}},V_{\mathrm{vac}})^\top.
$$

with penalized objective

$$
\sum_t
\Bigl(
y_t
-
X_{t,c}^\top(\beta+\delta_{s(t)})
-
\theta_{\mathrm{cont}(c)}
\Bigr)^2
+
\lambda_\beta \|\beta\|_2^2
+
\lambda_\delta \sum_s \|\delta_s\|_2^2 .
$$

- $\beta=(\beta_0,\ldots,\beta_4)$: global coefficient vector (light ridge penalty `lam_beta`)
- $\delta_s$: season-specific coefficient deviation (strong ridge penalty `lam_delta \gg lam_beta` -> (1) ensures identifiability (2) preserves temporal coherence)

**[B] AR(1)-increment** — exact augmented state $X_t=(S_t,S_{t-1})^\top$:

$$A(\rho)=\begin{pmatrix}1+\rho&-\rho\\1&0\end{pmatrix},\quad Q=\sigma^2\begin{pmatrix}1&0\\0&0\end{pmatrix},\quad C=(1,0)$$

**Observations:** Gaussian $y_t \mid S_t \sim N(\alpha_c + S_t,\, \tau^2)$ — exact Kalman filter throughout.

**rho M-step** — exact EM via smoother second moments:

$$\hat\rho = \frac{\sum_t \bigl(E[D_t D_{t-1}|Y] - \mu_t\,E[D_{t-1}|Y]\bigr)}{\sum_t E[D_{t-1}^2|Y]}$$

where $D_t = S_t - S_{t-1}$, using full $2\times2$ RTS cross-covariance $\mathrm{Cov}(X_t,X_{t-1}|Y)=G_{t-1}\Sigma^s_t$.


In [3]:
# Helper functions
def get_block_covariates(country, time_block):
    key = (country, time_block)
    if key in block_cov.index:
        return float(block_cov.loc[key, 'stringency_lag']), float(block_cov.loc[key, 'log_vac_lag'])
    return 0.0, 0.0


def compute_mu_t_c(df_c, beta, delta, theta_c):
    U_c1 = float(df_c['pc1'].iloc[0])
    U_c2 = float(df_c['pc2'].iloc[0])
    mu   = np.zeros(len(df_c))
    for i, (_, row) in enumerate(df_c.iterrows()):
        V_str, V_vac = get_block_covariates(df_c['country'].iloc[0], row['time_block'])
        X_t   = np.array([1., U_c1, U_c2, V_str, V_vac])
        d_s   = delta.get(row['time_block'], np.zeros(5))
        mu[i] = X_t @ (beta + d_s) + theta_c
    return mu


## Kalman filter

**M0/M1**: scalar state $S_t$.

**M2**: augmented state $X_t = (S_t, S_{t-1})^\top$ — exact AR(1) increment.

$$A(\rho) = \begin{pmatrix}1+\rho & -\rho \\ 1 & 0\end{pmatrix}, \quad
Q = \sigma^2\begin{pmatrix}1&0\\0&0\end{pmatrix}, \quad C=(1,0)$$

Observations are Gaussian throughout.


In [4]:
def kalman_filter(Y, log_pc, mu_seq, tau2, mu_s_1, Sigma_s_1,
                  sigma2=0.03,
                  ar1=False,
                  rho=0.0,
                  ):
    T  = len(Y)
    C  = np.array([1., 0.])
    Q2 = sigma2 * np.array([[1., 0.], [0., 0.]])

    innov_store = np.zeros(T)
    H_store     = np.zeros(T)

    # M2:
    if ar1:
        mu_f_2  = np.zeros((T, 2));  Sig_f_2 = np.zeros((T, 2, 2))
        mu_p_2  = np.zeros((T, 2));  Sig_p_2 = np.zeros((T, 2, 2))
        log_ml  = 0.0
        for t in range(T):
            if t == 0:
                mu_p_2[t]  = np.array([mu_s_1, mu_s_1])
                Sig_p_2[t] = np.diag([Sigma_s_1, Sigma_s_1])
            else:
                A = np.array([[1 + rho, -rho], [1., 0.]])
                mu_p_2[t]  = A @ mu_f_2[t-1] + np.array([mu_seq[t], 0.])
                Sig_p_2[t] = A @ Sig_f_2[t-1] @ A.T + Q2

            innov = Y[t] - log_pc - C @ mu_p_2[t]
            H_t   = float(C @ Sig_p_2[t] @ C) + tau2
            H_t   = max(H_t, 1e-8)

            innov_store[t] = innov
            H_store[t]     = H_t

            K_t = (Sig_p_2[t] @ C) / H_t
            mu_f_2[t]  = mu_p_2[t] + K_t * innov
            Sig_f_2[t] = (np.eye(2) - np.outer(K_t, C)) @ Sig_p_2[t]
            log_ml    += -0.5 * (np.log(2 * np.pi * H_t) + innov**2 / H_t)

        mu_f    = mu_f_2[:, 0];  Sigma_f = Sig_f_2[:, 0, 0]
        mu_p    = mu_p_2[:, 0];  Sigma_p = Sig_p_2[:, 0, 0]
        return (mu_f, Sigma_f, mu_p, Sigma_p, log_ml,
                mu_f_2, Sig_f_2, mu_p_2, Sig_p_2,
                innov_store, H_store)

    # M0 / M1:
    mu_f = np.zeros(T);  Sigma_f = np.zeros(T)
    mu_p = np.zeros(T);  Sigma_p = np.zeros(T)
    log_ml = 0.0
    for t in range(T):
        if t == 0:
            mu_p[t]    = mu_s_1;  Sigma_p[t] = Sigma_s_1
        else:
            mu_p[t]    = mu_f[t-1] + mu_seq[t];  Sigma_p[t] = Sigma_f[t-1] + sigma2

        innov = Y[t] - log_pc - mu_p[t]
        H_t   = Sigma_p[t] + tau2
        H_t   = max(H_t, 1e-8)

        innov_store[t] = innov
        H_store[t]     = H_t

        K_t        = Sigma_p[t] / H_t
        mu_f[t]    = mu_p[t] + K_t * innov
        Sigma_f[t] = (1 - K_t) * Sigma_p[t]
        log_ml    += -0.5 * (np.log(2 * np.pi * H_t) + innov**2 / H_t)

    return (mu_f, Sigma_f, mu_p, Sigma_p, log_ml,
            None, None, None, None,
            innov_store, H_store)


In [5]:
def rts_smoother(mu_f, Sigma_f, mu_p, Sigma_p,
                 mu_f_2=None, Sig_f_2=None, mu_p_2=None, Sig_p_2=None, rho=0.0):
    
    T             = len(mu_f)
    mu_s          = mu_f.copy();  Sigma_s = Sigma_f.copy()
    Sigma_s_cross = np.zeros(T - 1)

    # M2:
    if mu_f_2 is not None:
        A = np.array([[1 + rho, -rho], [1., 0.]])
        mu_s_2        = mu_f_2.copy()
        Sig_s_2       = Sig_f_2.copy()
        Sig_s_cross_2 = np.zeros((T - 1, 2, 2))

        for t in range(T - 2, -1, -1):
            Sig_p_inv        = np.linalg.pinv(Sig_p_2[t+1] + 1e-10 * np.eye(2))
            G_t              = Sig_f_2[t] @ A.T @ Sig_p_inv
            mu_s_2[t]        = mu_f_2[t]  + G_t @ (mu_s_2[t+1]  - mu_p_2[t+1])
            Sig_s_2[t]       = Sig_f_2[t] + G_t @ (Sig_s_2[t+1] - Sig_p_2[t+1]) @ G_t.T
            Sig_s_cross_2[t] = G_t @ Sig_s_2[t+1]

        mu_s    = mu_s_2[:, 0];  Sigma_s = Sig_s_2[:, 0, 0]
        for t in range(T - 2, -1, -1):
            Sigma_s_cross[t] = Sig_s_cross_2[t][0, 0]

        return mu_s, Sigma_s, Sigma_s_cross, mu_s_2, Sig_s_2, Sig_s_cross_2

    # M0/M1:
    for t in range(T - 2, -1, -1):
        G_t              = Sigma_f[t] / (Sigma_p[t+1] + 1e-10)
        mu_s[t]          = mu_f[t]    + G_t * (mu_s[t+1]    - mu_p[t+1])
        Sigma_s[t]       = Sigma_f[t] + G_t**2 * (Sigma_s[t+1] - Sigma_p[t+1])
        Sigma_s_cross[t] = G_t * Sigma_s[t+1]
    return mu_s, Sigma_s, Sigma_s_cross, None, None, None


## E-step


In [6]:
def e_step(df, alpha, beta, delta, theta, tau2,
           sigma2=0.03, ar1=False, rho=0.0):
    results = {}
    log_ml  = 0.0
    for country in countries:
        df_c   = df[df['country'] == country].sort_values('date').reset_index(drop=True)
        Y      = df_c['log_cases'].values.astype(float)
        log_pc = alpha[country]
        mu_seq = compute_mu_t_c(df_c, beta, delta, theta[country])

        out = kalman_filter(Y, log_pc, mu_seq, tau2, float(Y[0] - log_pc), sigma2,
                            sigma2=sigma2, ar1=ar1, rho=rho)
        (mu_f, Sigma_f, mu_p, Sigma_p, lml,
         mu_f_2, Sig_f_2, mu_p_2, Sig_p_2,
         innov_store, H_store) = out

        mu_s, Sigma_s, Sigma_s_cross, mu_s_2, Sig_s_2, Sig_s_cross_2 = rts_smoother(
            mu_f, Sigma_f, mu_p, Sigma_p, mu_f_2, Sig_f_2, mu_p_2, Sig_p_2, rho=rho)
        log_ml += lml

        results[country] = {
            'ES'            : mu_s,
            'ES2'           : Sigma_s + mu_s**2,
            'ESS'           : Sigma_s_cross + mu_s[1:] * mu_s[:-1],
            'Y'             : Y,
            'mu_seq'        : mu_seq,
            'PC_c1'         : float(df_c['pc1'].iloc[0]),
            'PC_c2'         : float(df_c['pc2'].iloc[0]),
            'blocks'        : df_c['time_block'].tolist(),
            'dates'         : df_c['date'].tolist(),
            'mu_s_2'        : mu_s_2,
            'Sig_s_2'       : Sig_s_2,
            'mu_p_2'        : mu_p_2,
            'Sig_p_2'       : Sig_p_2,
            'Sig_s_cross_2' : Sig_s_cross_2,
            'innov'         : innov_store,
            'H'             : H_store,
        }
    return results, log_ml


## M-step


In [7]:
def m_step(df, results, alpha, tau2,
           lam_beta=0.1,
           lam_delta=5.0,
           lam_theta=0.1,
           sigma2=0.03,
           ar1=False,
           rho=0.0,
           ):
    n_beta  = 5
    n_delta = n_blocks * 5
    n_theta = n_cont
    n_cols  = n_beta + n_delta + n_theta

    # ── [A] Beta + delta: block-structured ridge ──────────────────────────────
    rows_y, rows_x = [], []
    for country in countries:
        res   = results[country]
        df_c  = df[df['country'] == country].sort_values('date').reset_index(drop=True)
        T     = len(df_c)
        PC_c1 = res['PC_c1'];  PC_c2 = res['PC_c2']
        cont  = cont_map[country]

        if ar1 and res.get('mu_s_2') is not None:
            # D_t directly from augmented state: avoids differencing adjacent means
            delta_S = res['mu_s_2'][1:, 0] - res['mu_s_2'][1:, 1]
        else:
            delta_S = res['ES'][1:] - res['ES'][:-1]

        for t in range(T - 1):
            V_str, V_vac = get_block_covariates(country, df_c['time_block'].iloc[t+1])
            X_t  = np.array([1., PC_c1, PC_c2, V_str, V_vac])
            b    = df_c['time_block'].iloc[t+1]
            bi   = block_idx[b]

            row = np.zeros(n_cols)
            row[0:5]                          = X_t
            row[5 + bi*5: 5 + bi*5 + 5]      = X_t
            row[5 + n_delta + cont_idx[cont]] = 1.0

            rows_y.append(delta_S[t])
            rows_x.append(row)

    X_reg = np.array(rows_x);  y_reg = np.array(rows_y)
    penalty  = np.concatenate([np.full(n_beta,  lam_beta),
                                np.full(n_delta, lam_delta),
                                np.full(n_theta, lam_theta)])
    scale    = np.sqrt(penalty)
    X_scaled = X_reg / scale[np.newaxis, :]
    ridge    = Ridge(alpha=1.0, fit_intercept=False)
    ridge.fit(X_scaled, y_reg)
    coefs    = ridge.coef_ / scale

    beta_new  = coefs[:5]
    delta_new = {b: coefs[5 + block_idx[b]*5: 5 + block_idx[b]*5 + 5] for b in all_blocks}
    theta_raw = coefs[5 + n_delta:];  theta_raw -= theta_raw.mean()
    theta_new = {c: float(theta_raw[cont_idx[cont_map[c]]]) for c in countries}

    # ── [B] rho: exact EM update via smoother cross-covariances ──────────────
    rho_new = rho
    if ar1:
        c = np.array([1., -1.])   # D_t = c^T X_t
        num_rho = 0.0;  den_rho = 0.0

        for country in countries:
            res           = results[country]
            mu_s_2        = res.get('mu_s_2')
            Sig_s_2       = res.get('Sig_s_2')
            Sig_s_cross_2 = res.get('Sig_s_cross_2')
            mu_seq        = res['mu_seq']
            if mu_s_2 is None or Sig_s_cross_2 is None:
                continue
            T = mu_s_2.shape[0]

            for t in range(1, T):
                ED_lag_sq = (c @ Sig_s_2[t-1] @ c
                             + (c @ mu_s_2[t-1])**2)
                ED_cross  = (c @ Sig_s_cross_2[t-1] @ c
                             + (c @ mu_s_2[t]) * (c @ mu_s_2[t-1]))
                mu_t      = mu_seq[t]
                ED_lag    = c @ mu_s_2[t-1]
                num_rho  += ED_cross - mu_t * ED_lag
                den_rho  += ED_lag_sq

        rho_new = float(np.clip(num_rho / (den_rho + 1e-8), -0.95, 0.95))

    # ── tau2: exact Gaussian EM update (innovation-based) ────────────────────
    # Exact closed form for Gaussian model:
    #   tau2_new = (sum_t innov_t^2 - sum_t Sigma_p_t) / T_total
    # Derivation: d/d(tau2) E[log p(Y|S)] = 0 with Gaussian likelihood,
    # integrating out S using filter predictions.
    tau2_num = 0.0;  tau2_sp = 0.0;  N_tau = 0
    for country in countries:
        res   = results[country]
        innov = res['innov']
        H     = res['H']
        Sigma_p = np.maximum(H - tau2, 0.0)   # predicted state var: H_t - tau2

        tau2_num += float(np.sum(innov**2))
        tau2_sp  += float(np.sum(Sigma_p))
        N_tau    += len(innov)

    tau2_new = float(np.clip((tau2_num - tau2_sp) / max(N_tau, 1), 1e-6, 10.0))

    # ── alpha_c MAP ───────────────────────────────────────────────────────────
    alpha_new = {}
    for country in countries:
        Y   = results[country]['Y'];  ES = results[country]['ES'];  T_c = len(Y)
        alpha_new[country] = float(
            (np.sum(Y - ES) / tau2_new - 1.0) / (T_c / tau2_new + 1.0))

    return beta_new, delta_new, theta_new, tau2_new, alpha_new, rho_new


## EM runner


In [8]:
def run_EM(df, label, n_iter=300, lam_beta=0.1, lam_delta=5.0, lam_theta=0.1,
           tol=1e-4, ar1=False, rho_init=0.0):
           
    print(f"\n{'='*55}\n  {label}")
    print(f"  lam_beta={lam_beta}  lam_delta={lam_delta}  ar1={ar1}")
    print(f"{'='*55}")

    beta  = np.zeros(5)
    delta = {b: np.zeros(5) for b in all_blocks}
    theta = {c: 0.0 for c in countries}
    tau2  = 0.01
    alpha = {c: 0.0 for c in countries}
    rho   = rho_init
    log_mls = []

    for it in range(n_iter):
        results, log_ml = e_step(df, alpha, beta, delta, theta, tau2,
                                  sigma2=sigma2, ar1=ar1, rho=rho)
        log_mls.append(log_ml)

        beta, delta, theta, tau2, alpha, rho = m_step(
            df, results, alpha, tau2,
            lam_beta=lam_beta, lam_delta=lam_delta, lam_theta=lam_theta,
            sigma2=sigma2, ar1=ar1, rho=rho)

        if it % 50 == 0 or it == n_iter - 1:
            print(f"  iter {it+1:3d}  log_ml={log_ml:10.2f}  tau2={tau2:.5f}"
                  + (f"  rho={rho:.4f}" if ar1 else ""))

        if it > 0 and abs(log_mls[-1] - log_mls[-2]) < tol:
            print(f"  Converged at iteration {it+1}");  break

    return {'label': label, 'results': results,
            'beta': beta, 'delta': delta, 'theta': theta,
            'tau2': tau2, 'alpha': alpha, 'rho': rho,
            'log_mls': log_mls, 'final_log_ml': log_mls[-1]}


## Fit models

| Model | Added | `ar1` | Notes |
|---|---|---|---|
| **M0** | Baseline | `False` | No seasonal drift ($\delta_s=0$, $\lambda_\delta\to\infty$) |
| **M1** | + regularized $\delta_s$ | `False` | `lam_delta` large → borrows across seasons |
| **M2** | + AR(1)-increment | `True` | Exact 2-D augmented state, exact EM rho update |

All models: Gaussian observations, exact Kalman filter, exact EM M-step.


In [10]:
# M0: baseline
m0 = run_EM(df, label="M0 — baseline",
            lam_beta=0.1, lam_delta=1e6, ar1=False)

# M1: +[A] regularized seasonal drift
m1 = run_EM(df, label="M1 — +[A] seasonal drift delta_s",
            lam_beta=0.1, lam_delta=5.0, ar1=False)

# M2: +[A]+[B] AR(1) — exact Gaussian EM
m2 = run_EM(df, label="M2 — +[A]+[B] AR(1)",
            lam_beta=0.1, lam_delta=5.0, ar1=True, rho_init=0.0)

all_models = [m0, m1, m2]



  M0 — baseline
  lam_beta=0.1  lam_delta=1000000.0  ar1=False
  iter   1  log_ml= -30587.26  tau2=0.21510
  Converged at iteration 18

  M1 — +[A] seasonal drift delta_s
  lam_beta=0.1  lam_delta=5.0  ar1=False
  iter   1  log_ml= -30587.26  tau2=0.21510
  Converged at iteration 26

  M2 — +[A]+[B] AR(1)
  lam_beta=0.1  lam_delta=5.0  ar1=True
  iter   1  log_ml= -30587.26  tau2=0.21510  rho=0.6061
  Converged at iteration 37


In [12]:
for m in all_models:
    rho_val = m['rho'] if 'rho' in m else 0.0

    print(
        f"{m['label']}: "
        f"tau2={m['tau2']:.4f}, "
        f"rho={rho_val:.4f}"
    )

M0 — baseline: tau2=0.9287, rho=0.0000
M1 — +[A] seasonal drift delta_s: tau2=0.6111, rho=0.0000
M2 — +[A]+[B] AR(1): tau2=0.1800, rho=0.6202


## Save

In [13]:
with open(f'../data/all_models_{freq}.pkl', 'wb') as f:
    pickle.dump(all_models, f)
print(f"Saved all_models_{freq}.pkl")

for m in all_models:
    tag     = m['label'].split('—')[0].strip().lower().replace(' ', '_')
    records = []
    for country in countries:
        ES    = m['results'][country]['ES']
        dates = m['results'][country]['dates']
        for t in range(1, len(ES)):
            records.append({'country': country, 'date': dates[t],
                            'R_plug_in': np.exp(ES[t] - ES[t-1]), 'model': m['label']})
    pd.DataFrame(records).to_csv(f'../data/R_plugin_{freq}_{tag}.csv', index=False)
print("Saved R plug-in CSVs")


Saved all_models_seasonal.pkl
Saved R plug-in CSVs
